In [3]:
import os
import pickle
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm

# BatteryLife datasets
# CALB, CALCE, HNEI, HUST, ISU_ILCC, MATR, MICH, MICH_EXP, NA-ion, RWTH, SDU, SNL, Stanford_2, Tongji, UL_PUR, XJTU, ZN-coin

path = '/data/trf/python_works/BatteryLife/dataset/RWTH/'
ppath = Path(path)
files = os.listdir(path)
files = [i for i in files if i.endswith('.pkl')]

soc = []
cell = []
life_label = []
data = {}


def compress_stages(stage_list):
    """
    ['rest', 'rest', 'discharge', 'discharge', 'charge']
    -> ['rest', 'discharge', 'charge']
    """
    if len(stage_list) == 0:
        return []

    compressed = [stage_list[0]]
    for s in stage_list[1:]:
        if s != compressed[-1]:
            compressed.append(s)
    return compressed


def get_cycle_stage_description(
    current_series,
    nominal_capacity,
    zero_c_rate_threshold=0.02,
    remove_rest=True
):


    stage_seq = []

    for current in current_series:
        c_rate = current / nominal_capacity

        if abs(c_rate) <= zero_c_rate_threshold:
            stage = 'rest'
        elif c_rate > 0:
            stage = 'charge'
        else:
            stage = 'discharge'

        stage_seq.append(stage)

    compressed_stages = compress_stages(stage_seq)

    if remove_rest:
        compressed_stages = [s for s in compressed_stages]

    return compressed_stages


for file in tqdm(files):
    with open(path + f'{file}', 'rb') as f:
        cell_data = pickle.load(f)
        filename = file.split('.pkl')[0]
        length = len(cell_data['cycle_data'])
        capacity = cell_data['nominal_capacity_in_Ah']

        cycle_stage_summary = {}

        currents = []
        voltages = []
        times = []

        for i in range(0, length):

            cycle_data = cell_data['cycle_data'][i]
            cycle_df = pd.DataFrame()

            cycle_df['current'] = cycle_data['current_in_A']
            cycle_df['voltage'] = cycle_data['voltage_in_V']
            cycle_df['charge_capacity'] = cycle_data['charge_capacity_in_Ah']
            cycle_df['discharge_capacity'] = cycle_data['discharge_capacity_in_Ah']
            cycle_df['test time'] = cycle_data['time_in_s']
            cycle_df['cycle_number'] = cycle_data['cycle_number']

            current = cycle_df['current'].values.tolist()
            voltage = cycle_df['voltage'].values.tolist()
            time = cycle_df['test time'].values.tolist()

            stage_desc = get_cycle_stage_description(
                current_series=current,
                nominal_capacity=capacity,
                zero_c_rate_threshold=0.02,  
                remove_rest=True
            )

            cycle_stage_summary[i] = stage_desc
            discharge_count = len([i for i in stage_desc if i == 'discharge'])
            charge_count = len([i for i in stage_desc if i == 'charge'])
            if discharge_count > 1 or charge_count > 1:
                print(f'Cell {file}, Cycle {i+1} stages: {stage_desc}')

            currents += current
            voltages += voltage
            times += time


        # fig, ax1 = plt.subplots(figsize=(10, 6))

        # ax1.plot(times, voltages)
        # ax1.set_xlabel('Time (s)')
        # ax1.set_ylabel('Voltage (V)')

        # ax2 = ax1.twinx()
        # ax2.plot(times, currents)
        # ax2.set_ylabel('Current (A)')

        # plt.title(f'Voltage-Current vs Time Profile\nFile: {filename}\n')
        # plt.show()

  0%|          | 0/48 [00:00<?, ?it/s]

Cell RWTH_036.pkl, Cycle 709 stages: ['rest', 'discharge', 'rest', 'charge', 'discharge', 'rest', 'charge']
Cell RWTH_036.pkl, Cycle 1051 stages: ['rest', 'discharge', 'rest', 'charge', 'discharge', 'rest', 'charge']


  2%|▏         | 1/48 [00:02<02:17,  2.92s/it]

Cell RWTH_032.pkl, Cycle 799 stages: ['rest', 'discharge', 'rest', 'charge', 'discharge', 'rest', 'charge']
Cell RWTH_032.pkl, Cycle 1075 stages: ['rest', 'discharge', 'rest', 'charge', 'discharge', 'rest', 'charge']


  4%|▍         | 2/48 [00:06<02:23,  3.13s/it]

...

 96%|█████████▌| 46/48 [02:17<00:05,  2.94s/it]

Cell RWTH_007.pkl, Cycle 290 stages: ['rest', 'discharge', 'rest', 'charge', 'discharge', 'rest', 'charge']


 98%|█████████▊| 47/48 [02:21<00:03,  3.05s/it]

Cell RWTH_007.pkl, Cycle 2387 stages: ['rest', 'discharge', 'rest', 'charge', 'rest', 'discharge', 'rest', 'charge']
Cell RWTH_007.pkl, Cycle 2388 stages: ['rest', 'discharge', 'rest', 'charge', 'rest', 'discharge', 'rest', 'charge']
Cell RWTH_007.pkl, Cycle 2390 stages: ['rest', 'discharge', 'rest', 'charge', 'discharge', 'rest', 'charge']
Cell RWTH_007.pkl, Cycle 2391 stages: ['discharge', 'rest', 'charge', 'rest', 'discharge', 'rest', 'charge']
Cell RWTH_007.pkl, Cycle 2392 stages: ['rest', 'discharge', 'rest', 'charge', 'rest', 'discharge', 'rest', 'charge']
Cell RWTH_007.pkl, Cycle 2393 stages: ['rest', 'discharge', 'rest', 'charge', 'rest', 'discharge', 'rest', 'charge']
Cell RWTH_007.pkl, Cycle 2394 stages: ['rest', 'discharge', 'rest', 'charge', 'rest', 'discharge', 'rest', 'charge']
Cell RWTH_007.pkl, Cycle 2395 stages: ['rest', 'discharge', 'rest', 'charge', 'rest', 'discharge', 'rest', 'charge']
Cell RWTH_005.pkl, Cycle 1210 stages: ['rest', 'discharge', 'rest', 'charge', 'd

100%|██████████| 48/48 [02:24<00:00,  3.01s/it]
